# Annotated notebook

Explanatory notes have been added as Markdown cells immediately before the related code. The original code cells, their source text, outputs, execution counts and metadata have been preserved; no executable code has been edited. Run cells in dependency order, especially where a note flags that a variable or function is defined in a later cell.

### Check whether both model objects are available
Prints the observation counts stored in `model_A` and `model_B`. This is a quick check after a kernel restart; it works only after both models have been recreated.

In [5]:
print("Model A observations:", int(model_A.nobs))
print("Model B observations:", int(model_B.nobs))

NameError: name 'model_A' is not defined

### Check the current kernel state
Checks whether `df` exists and, if it does, whether it contains the `Archetype` column. This helps diagnose which objects survived a kernel restart.

In [13]:
print("df exists:", "df" in globals())
print("Archetype exists:", "Archetype" in df.columns if "df" in globals() else False)

df exists: False
Archetype exists: False


### Check the working directory and available files
Prints the current directory and its contents so the source CSV filename and saved outputs can be verified.

In [15]:
import os

print(os.getcwd())
print(os.listdir())

/mnt/c/Users/aakas/OneDrive/Desktop/Q3/Sports_biometric_DAI/Question-3
['.ipynb_checkpoints', 'Q3_Final_Comparison.csv', 'Q3_Model_Comparison.csv', 'Q3_Plots', 'Q3_Visualizations.pdf', 'Regression_and_sensitivity_analysis.ipynb']


### Reload the original dataset and required libraries
Imports pandas, NumPy and statsmodels' formula API, then reloads the original athlete-event CSV into `df`. This restores the raw data after a kernel restart.

In [16]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_csv("data_read_to_use.csv")

print("Dataset restored!")
print("Shape:", df.shape)
display(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'data_read_to_use.csv'

### Apply the archetype function (execution-order note)
This cell applies `classify_archetype` to every row. The function and its mapping must already exist in the kernel. In the visible notebook order, those definitions appear in cells 5 and 6 below, so execute cells 5 and 6 first, then run this cell—or use the application cell 7 after the definitions. The code itself is left unchanged.

In [17]:
df["Archetype"] = df.apply(classify_archetype, axis=1)

NameError: name 'df' is not defined

### Define the sport mapping
Maps sports whose archetype can be assigned directly from the sport name to the proposed functional categories. Event-dependent sports are handled separately in the classification function.

In [18]:
sport_mapping = {
    "Rowing": "Endurance",
    "Weightlifting": "Speed & Power",

    "Boxing": "Combat",
    "Wrestling": "Combat",
    "Judo": "Combat",

    "Shooting": "Precision",
    "Archery": "Precision",
    "Golf": "Precision",

    "Badminton": "Racket Sports",
    "Tennis": "Racket Sports",
    "Table Tennis": "Racket Sports",

    "Hockey": "Team Sports",
    "Football": "Team Sports",
    "Basketball": "Team Sports",

    "Gymnastics": "Technical / Mixed",
    "Diving": "Technical / Mixed",
    "Sailing": "Technical / Mixed",
    "Equestrianism": "Technical / Mixed"
}

### Define the event-aware archetype classifier
Defines the rules for Athletics, Swimming and Cycling using event descriptions or distances, then falls back to `sport_mapping` for other sports. This function must be run after the mapping dictionary is defined and before applying the classifier.

In [19]:
import re

def classify_archetype(row):

    sport = row["Sport"]
    event = str(row["Event"]).lower()

    if sport == "Athletics":

        if "decathlon" in event or "heptathlon" in event:
            return "Technical / Mixed"

        endurance_keywords = [
            "marathon",
            "kilometres walk",
            "10,000 metres",
            "5,000 metres",
            "3,000 metres",
            "1,500 metres",
            "800 metres"
        ]

        if any(word in event for word in endurance_keywords):
            return "Endurance"

        return "Speed & Power"

    elif sport == "Swimming":

        distance = int(re.search(r"\d+", event).group())

        return "Endurance" if distance >= 400 else "Speed & Power"

    elif sport == "Cycling":

        return "Speed & Power" if "sprint" in event else "Endurance"

    else:
        return sport_mapping.get(sport, pd.NA)

### Apply archetype labels and inspect the distribution
Applies the classification function to the complete dataset and prints the count of records in each archetype, including any unclassified rows.

In [8]:
df["Archetype"] = df.apply(
    classify_archetype,
    axis=1
)

print(df["Archetype"].value_counts(dropna=False))

Archetype
Team Sports          245
Speed & Power        157
Precision            144
Combat               115
Racket Sports         76
Endurance             61
Technical / Mixed     37
Name: count, dtype: int64


### Restore the decade variable and fit both pooled models
Creates `Year_Decade`, builds a common complete-case sample containing ID, Year, Height, Weight, Sex and Archetype, and fits Model A and Model B. Both use the same rows and athlete-ID clustered standard errors, making their year coefficients directly comparable.

In [9]:
df["Year_Decade"] = df["Year"] / 10

regression_data = df.dropna(
    subset=[
        "ID", "Year", "Height",
        "Weight", "Sex", "Archetype"
    ]
).copy()

model_A = smf.ols(
    "Weight ~ Height + Year_Decade",
    data=regression_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": regression_data["ID"]}
)

model_B = smf.ols(
    "Weight ~ Height + Year_Decade + C(Archetype) + C(Sex)",
    data=regression_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": regression_data["ID"]}
)

print("Both models restored successfully!")

Both models restored successfully!


### Compare Model A and Model B
Creates a table containing the year coefficient, p-value, R-squared and adjusted R-squared for each model. The coefficient is measured per decade because the predictor is `Year_Decade`.

In [10]:
comparison = pd.DataFrame({
    "Model A": [
        model_A.params["Year_Decade"],
        model_A.pvalues["Year_Decade"],
        model_A.rsquared,
        model_A.rsquared_adj
    ],
    "Model B": [
        model_B.params["Year_Decade"],
        model_B.pvalues["Year_Decade"],
        model_B.rsquared,
        model_B.rsquared_adj
    ]
}, index=[
    "Year Coefficient",
    "P-value",
    "R-squared",
    "Adjusted R-squared"
])

display(comparison.round(4))

,Model A,Model B
Year Coefficient,0.7144,0.9990
P-value,0.0453,0.0084
R-squared,0.4841,0.5526
Adjusted R-squared,0.4829,0.5477


### Calculate the change in the year coefficient
Calculates the absolute and percentage change from Model A's year coefficient to Model B's. This quantifies how the estimate changes after adding archetype and sex controls; it does not by itself establish causation.

In [11]:
beta_A = model_A.params["Year_Decade"]
beta_B = model_B.params["Year_Decade"]

print("Model A coefficient:", beta_A)
print("Model B coefficient:", beta_B)

print("Absolute change:", beta_B - beta_A)

print("Percentage change:",
      ((beta_B - beta_A) / abs(beta_A)) * 100)

Model A coefficient: 0.7144289427012839
Model B coefficient: 0.998961498263612
Absolute change: 0.28453255556232815
Percentage change: 39.826571763246235


### Select the Men's Hockey subset
Filters the common regression sample to male athletes in Hockey and copies those rows into `mens_hockey`. This subset is used for the within-sport sensitivity analysis.

In [12]:
mens_hockey = regression_data[
    (regression_data["Sport"] == "Hockey") &
    (regression_data["Sex"] == "M")
].copy()

### Inspect Men's Hockey sample coverage
Reports the number of event-level records, unique athlete IDs and Olympic editions in the subset, then lists the years represented. These checks help assess whether the subset has enough historical coverage for a time-trend model.

In [13]:
print("Total observations:", len(mens_hockey))

print("Unique athletes:", mens_hockey["ID"].nunique())

print("Olympic editions:", mens_hockey["Year"].nunique())

print(sorted(mens_hockey["Year"].unique()))

Total observations: 198
Unique athletes: 132
Olympic editions: 17
[np.int64(1948), np.int64(1952), np.int64(1956), np.int64(1960), np.int64(1964), np.int64(1968), np.int64(1972), np.int64(1976), np.int64(1980), np.int64(1984), np.int64(1988), np.int64(1992), np.int64(1996), np.int64(2000), np.int64(2004), np.int64(2012), np.int64(2016)]


### Fit the Men's Hockey sensitivity model
Checks that the subset contains at least three Olympic editions and ten unique athletes before fitting weight on height and decade-scaled year. Standard errors are clustered by athlete ID. If the coverage condition is not met, the cell prints a warning instead of fitting the model.

In [14]:
if (mens_hockey["Year"].nunique() >= 3 and
    mens_hockey["ID"].nunique() >= 10):

    hockey_model = smf.ols(
        "Weight ~ Height + Year_Decade",
        data=mens_hockey
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": mens_hockey["ID"]}
    )

    print(hockey_model.summary())

else:
    print("Insufficient data for a reliable sensitivity analysis.")

                            OLS Regression Results                            
Dep. Variable:                 Weight   R-squared:                       0.456
Model:                            OLS   Adj. R-squared:                  0.451
Method:                 Least Squares   F-statistic:                     40.15
Date:                Wed, 04 Nov 2026   Prob (F-statistic):           2.51e-14
Time:                        18:36:47   Log-Likelihood:                -611.44
No. Observations:                 198   AIC:                             1229.
Df Residuals:                     195   BIC:                             1239.
Df Model:                           2                                         
Covariance Type:              cluster                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept    -291.5433     60.301     -4.835      

### Extract the Men's Hockey year-trend results
Prints the year coefficient, p-value, 95% confidence interval, R-squared and observation count from the fitted hockey model.

In [15]:
print("Year coefficient:",
      hockey_model.params["Year_Decade"])

print("P-value:",
      hockey_model.pvalues["Year_Decade"])

print("95% Confidence interval:")

print(hockey_model.conf_int().loc["Year_Decade"])

print("R-squared:", hockey_model.rsquared)

print("Observations:", int(hockey_model.nobs))

Year coefficient: 1.3124468792464987
P-value: 5.1405486161650874e-05
95% Confidence interval:
0    0.677165
1    1.947729
Name: Year_Decade, dtype: float64
R-squared: 0.4560886712731018
Observations: 198


### Assemble the final three-model comparison
Combines Model A, Model B and Men's Hockey results into one table with year coefficient, p-value, R-squared and observations. This is the compact summary used in the report.

In [16]:
final_comparison = pd.DataFrame({
    "Model": ["Model A", "Model B", "Men's Hockey"],

    "Year_Coefficient": [
        model_A.params["Year_Decade"],
        model_B.params["Year_Decade"],
        hockey_model.params["Year_Decade"]
    ],

    "P_Value": [
        model_A.pvalues["Year_Decade"],
        model_B.pvalues["Year_Decade"],
        hockey_model.pvalues["Year_Decade"]
    ],

    "R_Squared": [
        model_A.rsquared,
        model_B.rsquared,
        hockey_model.rsquared
    ],

    "Observations": [
        int(model_A.nobs),
        int(model_B.nobs),
        int(hockey_model.nobs)
    ]
})

display(final_comparison.round(4))

,Model,Year_Coefficient,P_Value,R_Squared,Observations
0,Model A,0.7144,0.0453,0.4841,835
1,Model B,0.9990,0.0084,0.5526,835
2,Men's Hockey,1.3124,0.0001,0.4561,198


### Export the Q3 results and regression data
Writes the final three-model comparison and the pooled regression dataset to CSV files. These files preserve the numerical results and the data used for the pooled model fits.

In [17]:
final_comparison.to_csv(
    "Q3_Final_Comparison.csv",
    index=False
)

regression_data.to_csv(
    "Q3_Regression_Data.csv",
    index=False
)

print("Q3 analysis exported successfully!")

Q3 analysis exported successfully!


### Display the final comparison table
Displays the three-model comparison in notebook format for convenient inspection and copying into the report.

In [18]:
display(final_comparison)

,Model,Year_Coefficient,P_Value,R_Squared,Observations
0,Model A,0.714429,0.045283,0.484102,835
1,Model B,0.998961,0.008361,0.552555,835
2,Men's Hockey,1.312447,0.000051,0.456089,198


### Display the full Men's Hockey regression summary
Shows the complete OLS output, including coefficient estimates, clustered standard errors, test statistics, confidence intervals and diagnostic statistics.

In [19]:
print(hockey_model.summary())

                            OLS Regression Results                            
Dep. Variable:                 Weight   R-squared:                       0.456
Model:                            OLS   Adj. R-squared:                  0.451
Method:                 Least Squares   F-statistic:                     40.15
Date:                Wed, 04 Nov 2026   Prob (F-statistic):           2.51e-14
Time:                        18:38:08   Log-Likelihood:                -611.44
No. Observations:                 198   AIC:                             1229.
Df Residuals:                     195   BIC:                             1239.
Df Model:                           2                                         
Covariance Type:              cluster                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept    -291.5433     60.301     -4.835      

### Report row counts, unique athletes and Olympic coverage
Checks the pooled and Men's Hockey sample sizes, counts distinct athlete IDs and reports the number of Olympic editions represented. This distinguishes event-record counts from unique-athlete counts.

In [20]:
print("Full dataset:", len(regression_data))
print("Unique athletes:", regression_data["ID"].nunique())

print("Men's Hockey observations:", len(mens_hockey))
print("Men's Hockey athletes:", mens_hockey["ID"].nunique())
print("Men's Hockey Olympic editions:", mens_hockey["Year"].nunique())

Full dataset: 835
Unique athletes: 522
Men's Hockey observations: 198
Men's Hockey athletes: 132
Men's Hockey Olympic editions: 17


### Verify exported Q3 files
Lists files whose names begin with `Q3_`, confirming which CSV outputs are present in the working directory.

In [21]:
import os

print([f for f in os.listdir() if f.startswith("Q3_")])

['Q3_Final_Comparison.csv', 'Q3_Model_Comparison.csv', 'Q3_Model_Data.csv', 'Q3_Regression_Data.csv']


### Import plotting tools
Imports Matplotlib's plotting interface and the PDF backend used to combine figures into a single PDF document.

In [22]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

ModuleNotFoundError: No module named 'matplotlib'

### Install Matplotlib if it is missing
Uses the notebook's active Python environment to install Matplotlib. This is an environment setup step, not part of the statistical analysis.

In [23]:
%pip install matplotlib

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   -------- ------------------------------- 2.1/9.5 MB 10.2 MB/s eta 0:00:01
   -------------- ------------------------- 3.4/9.5 MB 10.0 MB/s eta 0:00:01
   ---------------------- ----------------- 5.2/9.5 MB 8.5 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.5 MB 9.2 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.5 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 8.4 MB/s  0:00:01
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------- ----------- 1.8/2.5 MB 9.2 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 6.7 MB/s  0:00:00
   ---------------------------------------- 0.0/7.3 MB ? eta -:--:--
   ------------ --------------------------- 2.4/7.3 MB 11.6 MB/s eta 0:00:01
   ------------------------ --------------- 4.5/7.3 MB 10.6 MB/s eta 0:00:01
   --------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


### Confirm plotting imports
Imports Matplotlib and the PDF backend again after installation and prints a confirmation message.

In [24]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

print("Matplotlib installed successfully!")

Matplotlib installed successfully!


### Attempt to collect currently open figures
Saves all Matplotlib figures that are still open in the current kernel into one PDF. If the kernel was restarted or figures were closed, the figure list can be empty and the PDF will contain no plots.

In [25]:
output_file = "All_Project_Plots.pdf"

figures = [plt.figure(n) for n in plt.get_fignums()]

with PdfPages(output_file) as pdf:
    for fig in figures:
        pdf.savefig(fig, bbox_inches="tight")

print("PDF saved!")
print("Total plots:", len(figures))

PDF saved!
Total plots: 0


### Generate and export the five Q3 visualisations
Creates a folder for individual PNGs and checks that the pooled regression data and all three fitted models exist. It then builds a single multi-page PDF containing: (1) yearly mean weight with approximate 95% standard-error bars, (2) year-coefficient estimates with model-based confidence intervals, (3) archetype composition by Olympic year, (4) Men's Hockey weight records with a fitted height-adjusted trend, and (5) weight distributions by archetype. Each figure is also saved as a high-resolution PNG. The yearly and composition charts describe the event-level records in `regression_data`; the Hockey line is evaluated at mean height.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import os

# Create output folder
os.makedirs("Q3_Plots", exist_ok=True)

# Check required variables
assert "regression_data" in globals(), "Regression data missing!"
assert "model_A" in globals(), "Model A missing!"
assert "model_B" in globals(), "Model B missing!"
assert "hockey_model" in globals(), "Hockey model missing!"

# Create one PDF containing all plots
with PdfPages("Q3_Visualizations.pdf") as pdf:

    # ==========================================
    # GRAPH 1: WEIGHT TREND OVER OLYMPIC YEARS
    # ==========================================

    yearly = regression_data.groupby("Year")["Weight"].agg(
        ["mean", "count", "std"]
    )

    yearly["SE"] = yearly["std"] / np.sqrt(yearly["count"])

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.errorbar(
        yearly.index,
        yearly["mean"],
        yerr=1.96 * yearly["SE"],
        marker="o",
        capsize=4,
        linewidth=2
    )

    ax.set_title("Average Athlete Weight Across Olympic Years")
    ax.set_xlabel("Olympic Year")
    ax.set_ylabel("Average Weight (kg)")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    pdf.savefig(fig)
    fig.savefig("Q3_Plots/01_Weight_Trend.png", dpi=300)
    plt.show()
    plt.close(fig)


    # ==========================================
    # GRAPH 2: MODEL A VS MODEL B
    # ==========================================

    models = [model_A, model_B, hockey_model]
    labels = ["Model A", "Model B", "Men's Hockey"]

    coefficients = [
        m.params["Year_Decade"] for m in models
    ]

    confidence_intervals = [
        m.conf_int().loc["Year_Decade"].values
        for m in models
    ]

    lower_errors = [
        coefficients[i] - confidence_intervals[i][0]
        for i in range(3)
    ]

    upper_errors = [
        confidence_intervals[i][1] - coefficients[i]
        for i in range(3)
    ]

    fig, ax = plt.subplots(figsize=(9, 6))

    ax.errorbar(
        coefficients,
        labels,
        xerr=[lower_errors, upper_errors],
        fmt="o",
        capsize=6,
        markersize=8
    )

    ax.axvline(0, linestyle="--", alpha=0.6)

    ax.set_title("Comparison of Year Coefficients")
    ax.set_xlabel("Estimated Weight Difference per Decade (kg)")
    ax.grid(axis="x", alpha=0.3)

    plt.tight_layout()
    pdf.savefig(fig)
    fig.savefig("Q3_Plots/02_Model_Comparison.png", dpi=300)
    plt.show()
    plt.close(fig)


    # ==========================================
    # GRAPH 3: ARCHETYPE COMPOSITION OVER TIME
    # ==========================================

    composition = pd.crosstab(
        regression_data["Year"],
        regression_data["Archetype"],
        normalize="index"
    ) * 100

    fig, ax = plt.subplots(figsize=(12, 7))

    composition.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        width=0.85
    )

    ax.set_title("Archetype Composition Across Olympic Years")
    ax.set_xlabel("Olympic Year")
    ax.set_ylabel("Percentage of Athlete-Event Records")
    ax.legend(
        title="Archetype",
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )

    plt.xticks(rotation=45)
    plt.tight_layout()
    pdf.savefig(fig)
    fig.savefig("Q3_Plots/03_Archetype_Composition.png", dpi=300)
    plt.show()
    plt.close(fig)


    # ==========================================
    # GRAPH 4: MEN'S HOCKEY WEIGHT TREND
    # ==========================================

    mens_hockey = regression_data[
        (regression_data["Sport"] == "Hockey") &
        (regression_data["Sex"] == "M")
    ].copy()

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.scatter(
        mens_hockey["Year"],
        mens_hockey["Weight"],
        alpha=0.45,
        label="Athlete-event records"
    )

    years = np.linspace(
        mens_hockey["Year"].min(),
        mens_hockey["Year"].max(),
        100
    )

    prediction_data = pd.DataFrame({
        "Height": [mens_hockey["Height"].mean()] * 100,
        "Year_Decade": years / 10
    })

    predicted_weights = hockey_model.predict(prediction_data)

    ax.plot(
        years,
        predicted_weights,
        linewidth=2.5,
        label="Adjusted trend at mean height"
    )

    ax.set_title("Men's Hockey: Weight Trend Across Olympic Years")
    ax.set_xlabel("Olympic Year")
    ax.set_ylabel("Weight (kg)")
    ax.legend()
    ax.grid(alpha=0.3)

    plt.tight_layout()
    pdf.savefig(fig)
    fig.savefig("Q3_Plots/04_Mens_Hockey_Trend.png", dpi=300)
    plt.show()
    plt.close(fig)


    # ==========================================
    # GRAPH 5: WEIGHT DISTRIBUTION BY ARCHETYPE
    # ==========================================

    archetypes = sorted(
        regression_data["Archetype"].dropna().unique()
    )

    weight_groups = [
        regression_data.loc[
            regression_data["Archetype"] == a, "Weight"
        ].dropna()
        for a in archetypes
    ]

    fig, ax = plt.subplots(figsize=(12, 7))

    ax.boxplot(
        weight_groups,
        tick_labels=archetypes,
        showmeans=True
    )

    ax.set_title("Athlete Weight Distribution by Archetype")
    ax.set_xlabel("Functional Archetype")
    ax.set_ylabel("Weight (kg)")
    plt.xticks(rotation=25)
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    pdf.savefig(fig)
    fig.savefig("Q3_Plots/05_Weight_Distribution.png", dpi=300)
    plt.show()
    plt.close(fig)


print("\nALL FIVE GRAPHS SAVED SUCCESSFULLY!")
print("PDF: Q3_Visualizations.pdf")
print("Individual PNGs: Q3_Plots folder")

AssertionError: Regression data missing!